# YOLO11s fine-tuning : like / dislike / palm

HaGRID 384p 샘플에서 3개 제스처만 뽑아 전이학습

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
%pip install -q ultralytics

## 1. 데이터셋

In [ ]:
!wget -q https://github.com/difficcd/YOLO-3class-tracking/releases/download/dataset-v1/gesture3.zip
!mkdir -p /content/gesture3 && unzip -q -o gesture3.zip -d /content/gesture3
yaml = '''path: /content/gesture3
train: images/train
val: images/val
test: images/test
names:
  0: like
  1: dislike
  2: palm
'''
open('/content/gesture3/data.yaml', 'w').write(yaml)
!for s in train val test; do echo $s $(ls /content/gesture3/images/$s | wc -l); done

## 2. 학습
COCO 사전학습 yolo11s.pt 에서 시작 → 헤드를 3클래스로 바꿔서 학습

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11s.pt')
model.train(data='/content/gesture3/data.yaml', epochs=40, imgsz=512, batch=32,
            patience=10, project='/content/runs', name='gesture3', exist_ok=True)

## 3. test 셋 평가

In [ ]:
best = YOLO('/content/runs/gesture3/weights/best.pt')
m = best.val(data='/content/gesture3/data.yaml', split='test', imgsz=512,
             project='/content/runs', name='test', exist_ok=True)
print('mAP50', m.box.map50, 'mAP50-95', m.box.map)
for i, n in best.names.items():
    print(n, 'P=%.3f R=%.3f AP50=%.3f' % (m.box.p[i], m.box.r[i], m.box.ap50[i]))

## 4. 결과 다운로드

In [ ]:
!cd /content/runs && zip -qr /content/runs.zip gesture3 test
from google.colab import files
files.download('/content/runs.zip')